# 69 — Latency Profiling
**Goal:** Profile pipeline latency — find bottlenecks in parsing, embedding, and LLM calls.

Accuracy is only half of production readiness: a resume pipeline that scores 99% F1 but takes 30 seconds per document is a demo, not a product. This chapter closes the Evaluation block with **latency profiling** — measuring where time actually goes across parsing, section detection, extraction, embedding, search, and LLM calls — then shows two profiling techniques and a bottleneck-by-bottleneck optimization playbook.

**Why it matters for resumes / ATS:** batch resume processing has real throughput requirements (thousands of documents, cost per call), and interactive products have real response-time budgets (a candidate or recruiter waiting on a rewrite). Profiling replaces guesswork: the data decides whether you spend engineering effort on caching embeddings or on swapping the PDF parser. It also quantifies the price of the LLM step earlier chapters added — the rewrite that improves quality has a latency cost, and you should know it.

## 1. Why Latency Matters

Latency is not one number — it is a budget across pipeline stages, and the budget is dominated by a few expensive stages. The code prints an illustrative per-stage budget: PDF parsing ~200ms, OCR ~2–5s when needed, section detection ~10ms, regex skill extraction ~5ms, normalization ~50ms, embedding ~100ms, LLM rewriting ~1–3s, and FAISS search ~10ms at 1M vectors. The totals tell the product story: roughly 300ms without an LLM step, 2–4s with one.

**What the code does:** prints the budget table and flags the three bottlenecks — OCR, LLM calls, and embedding generation. Read the numbers by *order of magnitude*: OCR and LLM calls live in seconds, everything else in milliseconds. That single observation dictates the entire optimization strategy of this chapter: optimize the seconds first.

**Try it:** sanity-check your own pipeline against this budget — if your section detection or regex extraction shows up in the seconds range, something is structurally wrong, not merely slow.

In [ ]:
print('''Resume analysis pipeline latency budget:
- PDF parsing: ~200ms (pdfplumber)
- OCR: ~2-5s (if needed)
- Section detection: ~10ms
- Skill extraction (regex): ~5ms
- Skill normalization: ~50ms
- Embedding: ~100ms (sentence-transformers)
- LLM rewriting: ~1-3s (gpt-4o-mini)
- FAISS search: ~10ms (1M vectors)

Total: ~300ms (no LLM) or ~2-4s (with LLM)
Bottlenecks: OCR, LLM calls, embedding generation''')

## 2. Simple Profiling

The first profiling tool is `time.time()` around each stage — coarse, but it answers the only question that matters: which stage owns the runtime? The code wraps four stages in timers: normalization, simulated section detection (a 5ms sleep), regex skill extraction, and a simulated 50ms embedding step, then prints each stage's time and share of the total.

**What the code does:** running it yields roughly 60ms total, with the simulated embedding at ~51ms — about 85% of runtime — and section detection ~9ms; normalization and skill extraction are below timer resolution. One real finding worth noting: the printed percentages come out inflated (embedding shows as thousands of percent) because the cell computes `pct = t / total * 100` and then formats with `:.0%`, which scales by 100 a second time. The relative ordering is still the lesson — embedding dominates, everything else is noise.

**Try it:** delete the two `time.sleep()` calls and rerun — the remaining stages are so fast that timer noise dominates, which is exactly when you need the function-level profiler in the next section.

In [ ]:
import time

def profile_pipeline(text):
    """Profile each stage of the pipeline."""
    times = {}
    
    # Stage 1: Text normalization
    t0 = time.time()
    normalized = text.lower().strip()
    times['normalization'] = time.time() - t0
    
    # Stage 2: Section detection (simulated)
    t0 = time.time()
    sections = {"summary": True, "skills": True}
    time.sleep(0.005)  # Simulate work
    times['section_detect'] = time.time() - t0
    
    # Stage 3: Skill extraction (regex)
    t0 = time.time()
    import re
    skills = re.findall(r"\\b(Python|Java|SQL|AWS|NLP)\\b", text, re.IGNORECASE)
    times['skill_extract'] = time.time() - t0
    
    # Stage 4: Embedding (simulated)
    t0 = time.time()
    time.sleep(0.05)  # Simulate 50ms embedding
    times['embedding'] = time.time() - t0
    
    total = sum(times.values())
    print(f"Pipeline profile ({total*1000:.0f}ms total):")
    for stage, t in times.items():
        pct = t / total * 100
        print(f"  {stage:20s} {t*1000:6.1f}ms ({pct:.0%})")
    return times

text = "Python developer with 5 years NLP and AWS experience"
profile_pipeline(text)

## 3. Profiling with cProfile

Wall-clock timing around stages tells you *which stage* is slow; `cProfile` tells you *which function* inside it is slow. It records every function call with its cumulative time, and `pstats` formats the result sorted by `cumtime` — total time spent in a function including everything it calls. That sort order is the right default: it surfaces the true cost of a call chain, not just a leaf function's own time.

**What the code does:** profiles `slow_function()` — 1000 iterations of `sum(range(i * 100))`. The profile reports roughly 2000 function calls dominated by the built-in `sum` and `range` invocations inside the loop (the observed runtime here was ~1.65s on the venv machine; the number varies by hardware, the structure does not). The takeaway: even trivial pure-Python work shows up clearly in cProfile, so there is no excuse for guessing where time goes in a real pipeline.

**Try it:** point the same `cProfile.Profile()` pattern at `profile_pipeline()` from the previous section — the simulated sleeps now appear as real wall time attributed to their call sites.

In [ ]:
import cProfile, pstats, io

def slow_function():
    """A slow function we want to profile."""
    result = []
    for i in range(1000):
        result.append(sum(range(i * 100)))
    return result

# Profile
profiler = cProfile.Profile()
profiler.enable()
slow_function()
profiler.disable()

s = io.StringIO()
ps = pstats.Stats(profiler, stream=s).sort_stats('cumtime')
ps.print_stats(10)
print("cProfile output (top 10 by cumulative time):")
print(s.getvalue())

## 4. Optimizing Bottlenecks

Optimization rule: profile first, then fix the biggest bottleneck, then profile again — never optimize blind. The code prints the playbook for the three expensive stages identified in Section 1. For **PDF parsing**: cache parsed text keyed by file hash, prefer PyMuPDF over pdfplumber, and parse multiple documents in parallel. For **embedding**: precompute and cache all resume embeddings, swap to a smaller model (MiniLM instead of MPNet) when quality allows, and batch encode with `model.encode(list_of_texts)` instead of looping single calls. For **LLM calls**: stream for real-time apps, cap `max_tokens` low for extraction tasks, cache common queries, and route simple tasks to cheaper models.

**What the code does:** prints the three strategies. Two patterns recur across every stage — **caching** (never recompute what you already computed) and **batching** (amortize fixed costs over many items). Together they typically buy an order of magnitude before any algorithmic change is needed.

**Try it:** apply the rules to the Section 2 profile: the embedding stage is the bottleneck, so the highest-leverage fix is caching embeddings, not micro-optimizing the regex.

In [ ]:
print('''Bottleneck optimization strategies:
1. PDF parsing:
   - Cache parsed text by file hash
   - Use PyMuPDF (faster than pdfplumber)
   - Parallel parse multiple documents

2. Embedding:
   - Precompute and cache all resume embeddings
   - Use smaller models (MiniLM vs MPNet)
   - Batch encode (model.encode(list_of_texts))

3. LLM calls:
   - Use streaming for real-time apps
   - Set low max_tokens for extraction tasks
   - Cache common queries
   - Use cheaper models for simple tasks''')

## Summary: Profile before optimizing. Target the biggest bottleneck first. Cache aggressively.

**Measure before you optimize — and let the profile, not intuition, pick the bottleneck.**

The pipeline's latency is dominated by a handful of stages — OCR, LLM calls, embedding — and every optimization playbook in this chapter (caching, batching, cheaper models, faster parsers) targets those, not the millisecond stages profiling shows are already cheap. Simple wall-clock timing finds the stage; cProfile finds the function; the percent-formatting quirk in Section 2 is a reminder that profiling output deserves scrutiny too. This closes the Evaluation block: accuracy metrics, error analysis, hallucination guards, prompt A/B testing, human agreement, and now latency — the complete checklist for judging whether the resume pipeline is correct, trustworthy, and fast enough to ship.